# Day 1, hands-on 1: the three questions, worked

Three counting questions on the same thirty orders, answered with a loop, a condition and two accumulators.

Every placeholder is filled with the option the answer key records, and the notebook is executed
from a clean kernel so every output and every check is visible on the page. The line under each
step says why the other three letters fail.

Where this sits in the day, and the steps this notebook walks.

In [1]:
import sys, pathlib

here = pathlib.Path.cwd()
for parent in [here, *here.parents]:
    if (parent / "scripts" / "c2kit.py").exists():
        sys.path.insert(0, str(parent / "scripts"))
        break
import c2kit as kit

kit.side_by_side(
    kit.ladder(["kernel and types", "records and lookups", "hands-on: the three questions", "hands-on: find the mistake"], lit=2, title="the day's notebooks", show=False),
    kit.flow(["the Student segment", "the returned orders", "delivered above Rs 2,000", "the overlap"], title="this notebook's steps", show=False),
)

## Setup

The same thirty orders the teaching notebooks read, from the same file in `../data/`.

In [2]:
records = kit.load_records("C2_W01_D01_orders_STUDENT.py")

print(len(records), "Kalpa Retail orders loaded from ../data/")
print(records[0])

30 Kalpa Retail orders loaded from ../data/
{'order_id': 'KR4200', 'segment': 'Retail-Core', 'amount': '4500', 'status': 'returned', 'order_date': '2026-08-03'}


## Step 1. The Student segment

Count the Student orders and total what they are worth. Two accumulators, one walk, and the condition decides which orders count.

In [3]:
kit.flow(["the Student segment", "the returned orders", "delivered above Rs 2,000", "the overlap"], lit=0)

In [4]:
# TODO 1. Which condition selects the Student orders?
#   a) r["segment"] is "Student"
#   b) r["segment"] == "Student"
#   c) r["segment"] = "Student"
#   d) r["Student"] == r["segment"]

# TODO 2. What goes into the total accumulator?
#   a) int(r["amount"])
#   b) r["amount"]
#   c) str(r["amount"])
#   d) r["total"]
count = 0
total = 0
for r in records:
    if r["segment"] == "Student":
        count = count + 1
        total = total + int(r["amount"])

print(count, total)

7 12945


In [5]:
kit.check("seven Student orders", count == 7, f"count is {count}")
kit.check("they total Rs 12,945", total == 12945, f"total is {total}")

Option a asks whether two strings are the same object rather than the same text, and it happens to work on short strings for a reason that has nothing to do with your data. Option c is an assignment inside a condition, which Python rejects outright. Option d looks up a key called `Student`, which no record carries. For TODO 2, option b hands the accumulator a string on KR4200 and stops the loop, option c makes every amount text so the addition fails, and option d names a field that does not exist.

## Step 2. The returned orders

The same shape with a different condition. This is the question that walks into KR4200, whose amount arrived as text, so the conversion is doing real work here rather than being a formality.

In [6]:
kit.flow(["the Student segment", "the returned orders", "delivered above Rs 2,000", "the overlap"], lit=1)

In [7]:
# TODO 3. Which expression names the first returned order in the file?
#   a) records[0]["order_id"] if records[0]["status"] == "returned" else None
#   b) [r["order_id"] for r in records][0]
#   c) records["returned"][0]
#   d) min(r["order_id"] for r in records)
count = 0
total = 0
for r in records:
    if r["status"] == "returned":
        count = count + 1
        total = total + int(r["amount"])

first_returned = records[0]["order_id"] if records[0]["status"] == "returned" else None
print(count, total, first_returned)

7 13670 KR4200


In [8]:
kit.check("seven returned orders", count == 7, f"count is {count}")
kit.check("they total Rs 13,670", total == 13670, f"total is {total}")
kit.check("KR4200 is the first returned order, and its amount is text",
          first_returned == "KR4200" and isinstance(records[0]["amount"], str))

Option b returns the first order in the file whatever its status, which is the same answer today by accident and wrong the moment the file is re-sorted. Option c indexes a list with a string, which raises. Option d returns the smallest id in the whole file, ignoring status entirely.

## Step 3. Delivered orders above Rs 2,000

Two conditions have to hold at once. Write them as one condition with `and`, so the reader sees both requirements in one place.

In [9]:
kit.flow(["the Student segment", "the returned orders", "delivered above Rs 2,000", "the overlap"], lit=2)

In [10]:
# TODO 4. Which condition keeps delivered orders above Rs 2,000?
#   a) r["status"] == "delivered" or int(r["amount"]) > 2000
#   b) r["status"] == "delivered" and r["amount"] > 2000
#   c) r["status"] == "delivered" and int(r["amount"]) > 2000
#   d) int(r["amount"]) > 2000 and r["status"] != "returned"
count = 0
total = 0
for r in records:
    if r["status"] == "delivered" and int(r["amount"]) > 2000:
        count = count + 1
        total = total + int(r["amount"])

print(count, total)

6 15520


In [11]:
kit.check("six orders are delivered and above Rs 2,000", count == 6, f"count is {count}")
kit.check("they total Rs 15,520", total == 15520, f"total is {total}")
kit.check("that is fewer than the thirteen delivered and the thirteen above Rs 2,000",
          count < 13)

Option a uses `or`, so it keeps every delivered order and every large order and counts twenty of them. Option b compares a text amount against a number on KR4200 and raises `TypeError`. Option d lets cancelled orders through, because not returned is not the same as delivered.

## Step 4. Why the three counts do not add to thirty

Your three answers are 7, 7 and 6, which come to 20 across thirty orders. Work out whether that is allowed by finding an order that lands inside two of the three answers.

In [12]:
kit.flow(["the Student segment", "the returned orders", "delivered above Rs 2,000", "the overlap"], lit=3)

In [13]:
# TODO 5. Which expression finds the orders in both the Student and the returned answers?
#   a) student + returned
#   b) student & returned
#   c) student - returned
#   d) student == returned
student = {r["order_id"] for r in records if r["segment"] == "Student"}
returned = {r["order_id"] for r in records if r["status"] == "returned"}
large_delivered = {r["order_id"] for r in records
                   if r["status"] == "delivered" and int(r["amount"]) > 2000}

overlap = student & returned
print(sorted(overlap))
print("in none of the three:", 30 - len(student | returned | large_delivered))

['KR4203', 'KR4221']
in none of the three: 14


In [14]:
kit.check("some orders land in two answers at once", len(overlap) >= 1,
          f"{len(overlap)} orders are both Student and returned")
kit.check("no order is both returned and delivered",
          len(returned & large_delivered) == 0)

## What to post

Post one line with the five letters in order, then the three pairs of numbers:

```
1b 2a 3a 4c 5b
Student 7 / 12945
Returned 7 / 13670
Delivered above 2000 6 / 15520
```

Then one sentence saying whether three counts that add to twenty across thirty orders is a
problem, and naming one order that lands in two of them.

In [15]:
kit.flow(["the Student segment", "the returned orders", "delivered above Rs 2,000", "the overlap"], lit=3, title="the notebook, end to end")
kit.check_summary()